# RFT-0002 — R1 native self-RFT rollout, 3-session parallel A100

이 노트북은 기존 `RFT-0001` 2,000문항 pilot을 재사용하고, 고정
`tune/dev/test` holdout과 숫자-정규화 template이 겹치지 않는 **나머지 공식
clean-train 문항**만 Qwen으로 4회씩 풀이합니다.

세 개의 Colab 세션에서 동일한 노트북을 열고 Cell 2의 `SHARD_INDEX`만 각각
`0`, `1`, `2`로 바꿔 실행합니다. 각 세션은 별도 JSONL/CSV에 기록하므로 동시에
실행해도 충돌하지 않으며, 중단 후 같은 shard index로 다시 실행하면 완료 ID를
건너뛰고 이어집니다.

생성 정책은 챔피언 R1 경로에 맞춥니다.

- base: `Qwen/Qwen2.5-3B-Instruct`
- 문제당 4회, temperature 0.8, top-p 0.95, 최대 2,048 새 토큰
- 마지막 `\\boxed{INTEGER}`가 공식 답과 일치한 풀이만 후보
- 문항당 최대 2개, 중복·장문·자기모순 제거
- 외부 API, Pro4, leaderboard/test는 사용하지 않음

모든 세션이 끝난 뒤 한 세션에서 Cell 9의 `RUN_FINAL_MERGE=True`로 바꾸고
Cell 9~10을 실행해 pilot과 세 shard를 병합합니다.

In [ ]:
# Cell 1 — Install vLLM in a fresh A100 runtime.
# vLLM owns the compatible Torch/Transformers stack for this inference-only notebook.
%pip install -q --no-cache-dir \
  "nvidia-cuda-runtime==13.0.88" \
  "nvidia-cuda-nvrtc==13.0.88" \
  "vllm==0.26.0" \
  "pandas>=2.2,<3"
# torchcodec is optional for vLLM text generation and its Colab wheel currently
# tries to load an incompatible video extension. Remove it after vLLM installation.
%pip uninstall -q -y torchcodec

In [ ]:
# Cell 2 — EDIT ONLY SHARD_INDEX in the three sessions.
# Session A: 0, Session B: 1, Session C: 2
SHARD_INDEX = 0
NUM_SHARDS = 3

assert SHARD_INDEX in range(NUM_SHARDS), (SHARD_INDEX, NUM_SHARDS)

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "main"
RUN_ID = "RFT-0002-r1-native-k4-full"
SEED = 20260823

N_SAMPLES = 4
TEMPERATURE = 0.8
TOP_P = 0.95
INITIAL_MAX_NEW_TOKENS = 2048
EXTENDED_MAX_NEW_TOKENS = 4096
VLLM_MAX_MODEL_LEN = 8192

# vLLM continuous batching settings for one A100 40GB.
# The engine, not Python, dynamically schedules up to MAX_NUM_SEQS active sequences.
CHUNK_PROMPTS = 256
GPU_MEMORY_UTILIZATION = 0.94
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 65536

PROMPT_VERSION = "champion_r1_boxed_k4_v1"
SYSTEM_PROMPT = "You are a helpful assistant that solves math problems step by step."
USER_SUFFIX = (
    "Solve this step by step, then give the final answer as a single integer "
    "inside \\boxed{}."
)

print(f"[CONFIG] shard={SHARD_INDEX}/{NUM_SHARDS - 1} "
      f"n={N_SAMPLES} chunk={CHUNK_PROMPTS} "
      f"max_new={INITIAL_MAX_NEW_TOKENS}->{EXTENDED_MAX_NEW_TOKENS}")

In [ ]:
# Cell 3 — Mount Drive, resolve Unicode-safe paths, and locate immutable inputs.
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import hashlib
import json
import math
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import drive

MOUNT_ROOT = Path("/content/drive")
if not (MOUNT_ROOT / "MyDrive").exists():
    drive.mount(str(MOUNT_ROOT))
DRIVE_ROOT = MOUNT_ROOT / "MyDrive"
assert DRIVE_ROOT.exists(), DRIVE_ROOT

def canonical_name(value):
    return unicodedata.normalize("NFC", str(value)).strip().casefold()

project_names = {
    canonical_name("2026소중한챌린지"),
    canonical_name("2026_소중한챌린지"),
}
project_matches = [
    path for path in DRIVE_ROOT.iterdir()
    if path.is_dir() and canonical_name(path.name) in project_names
]
assert len(project_matches) == 1, f"Expected one project folder: {project_matches}"
PROJECT_DIR = project_matches[0]
RUNS_DIR = PROJECT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Prefer project/data/clean_train.csv. No leaderboard/test filename is searched or read.
preferred_clean = PROJECT_DIR / "data" / "clean_train.csv"
if preferred_clean.exists():
    CLEAN_TRAIN_PATH = preferred_clean
else:
    clean_candidates = [
        path for path in PROJECT_DIR.rglob("clean_train.csv")
        if "runs" not in {canonical_name(part) for part in path.parts}
    ]
    assert clean_candidates, "clean_train.csv not found."
    CLEAN_TRAIN_PATH = min(clean_candidates, key=lambda p: (len(p.parts), len(str(p))))

# Reuse the exact immutable grouped split used by the pilot when present.
known_split_dir = (
    RUNS_DIR / "AUDIT-0002-clean-split-passN-20260821-215706" / "splits"
)
split_dirs = sorted(
    [
        path for path in RUNS_DIR.glob("AUDIT-0002-clean-split-passN-*/splits")
        if all((path / name).exists() for name in [
            "tune_v1.csv", "dev_v1.csv", "test_v1.csv", "split_manifest.json"
        ])
    ],
    key=lambda path: path.parent.name,
)
assert split_dirs, "AUDIT-0002 immutable split not found."
SOURCE_SPLIT_DIR = known_split_dir if all(
    (known_split_dir / name).exists() for name in [
        "tune_v1.csv", "dev_v1.csv", "test_v1.csv", "split_manifest.json"
    ]
) else split_dirs[-1]

# Reuse the completed 2K pilot. Prefer the known run, otherwise newest complete run.
known_pilot = RUNS_DIR / "RFT-0001-phase1-pilot-20260822-083735"
pilot_runs = []
if (known_pilot / "data" / "pilot_2000.csv").exists():
    pilot_runs.append(known_pilot)
pilot_runs += sorted(
    [
        path for path in RUNS_DIR.glob("RFT-0001-phase1-pilot-*")
        if (path / "data" / "pilot_2000.csv").exists()
        and (path / "data" / "rft_pilot_strict.csv").exists()
    ],
    key=lambda path: path.name,
)
# Deduplicate paths while preserving order; choose the known run, else newest complete.
pilot_runs = list(dict.fromkeys(pilot_runs))
assert pilot_runs, "Completed RFT-0001 pilot artifacts not found."
PILOT_RUN_DIR = known_pilot if known_pilot in pilot_runs else pilot_runs[-1]
PILOT_INDEX_PATH = PILOT_RUN_DIR / "data" / "pilot_2000.csv"
PILOT_VERIFIED_PATH = PILOT_RUN_DIR / "data" / "rft_pilot_strict.csv"
assert PILOT_VERIFIED_PATH.exists(), PILOT_VERIFIED_PATH

EXP_DIR = RUNS_DIR / RUN_ID
DATA_DIR = EXP_DIR / "data"
CAND_DIR = EXP_DIR / "candidates"
REPORT_DIR = EXP_DIR / "reports"
for path in [DATA_DIR, CAND_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SHARD_TAG = f"shard_{SHARD_INDEX:02d}_of_{NUM_SHARDS:02d}"
SHARD_INPUT_PATH = DATA_DIR / f"remaining_{SHARD_TAG}.csv"
RAW_ROLLOUT_PATH = CAND_DIR / f"r1_native_rollouts_{SHARD_TAG}.jsonl"
SHARD_AUDIT_PATH = DATA_DIR / f"candidate_audit_{SHARD_TAG}.csv"
SHARD_VERIFIED_PATH = DATA_DIR / f"r1_native_verified_{SHARD_TAG}.csv"
SHARD_REPORT_PATH = REPORT_DIR / f"rollout_report_{SHARD_TAG}.json"

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert torch.cuda.is_available(), "Select an A100 GPU runtime."
free_bytes, total_bytes = torch.cuda.mem_get_info()
print("[PATH] project:", PROJECT_DIR)
print("[PATH] clean:", CLEAN_TRAIN_PATH)
print("[PATH] split:", SOURCE_SPLIT_DIR)
print("[PATH] pilot:", PILOT_RUN_DIR)
print("[PATH] experiment:", EXP_DIR)
print("[PATH] raw shard output:", RAW_ROLLOUT_PATH)
print("[GPU]", torch.cuda.get_device_name(0),
      f"free={free_bytes/1024**3:.1f}GiB total={total_bytes/1024**3:.1f}GiB")

In [ ]:
# Cell 4 — Rebuild the leakage-free training pool and deterministic remaining shard.
MANUAL_EXCLUDE_IDS = {
    "train-011240", "train-001203", "train-003199", "train-008274",
    "train-012773", "train-009597", "train-011834",
}

def read_csv_strings(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    frame.columns = [str(column).lstrip("\ufeff").strip() for column in frame.columns]
    return frame.fillna("").astype(str)

def normalize_integer(value):
    match = re.fullmatch(r"\s*(-?\d+)\s*", str(value or ""))
    return str(int(match.group(1))) if match else None

def normalized_template(text):
    text = unicodedata.normalize("NFC", str(text)).casefold()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\d+(?:\.\d+)?", "#", text)
    return text.strip()

clean = read_csv_strings(CLEAN_TRAIN_PATH)
assert {"id", "question", "answer"}.issubset(clean.columns), clean.columns.tolist()
clean = clean[["id", "question", "answer"]].copy()
clean["answer"] = clean["answer"].map(normalize_integer)
assert clean["id"].is_unique
assert clean["question"].ne("").all()
assert clean["answer"].notna().all()
clean["template_key"] = clean["question"].map(normalized_template)

split_frames = {}
for split_name in ["tune", "dev", "test"]:
    frame = read_csv_strings(SOURCE_SPLIT_DIR / f"{split_name}_v1.csv")
    assert {"id", "question", "answer"}.issubset(frame.columns)
    if "template_key" not in frame.columns:
        frame["template_key"] = frame["question"].map(normalized_template)
    split_frames[split_name] = frame

holdout_ids = set().union(*(set(frame["id"]) for frame in split_frames.values()))
holdout_templates = set().union(*(
    set(frame["template_key"]) for frame in split_frames.values()
))
assert len(holdout_ids) == sum(len(frame) for frame in split_frames.values())

training_pool = clean.loc[
    ~clean["id"].isin(holdout_ids | MANUAL_EXCLUDE_IDS)
    & ~clean["template_key"].isin(holdout_templates)
].copy()

pilot_index = read_csv_strings(PILOT_INDEX_PATH)
assert {"id", "question", "answer"}.issubset(pilot_index.columns)
pilot_ids = set(pilot_index["id"])
assert len(pilot_ids) == len(pilot_index), "Duplicate pilot IDs."
assert pilot_ids.issubset(set(training_pool["id"])), "Pilot contains non-training IDs."

remaining = training_pool.loc[~training_pool["id"].isin(pilot_ids)].copy()

def shard_for_id(qid):
    payload = f"{SEED}|{qid}".encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest(), 16) % NUM_SHARDS

remaining["shard_index"] = remaining["id"].map(shard_for_id)
shard_frame = (
    remaining.loc[remaining["shard_index"] == SHARD_INDEX]
    .sort_values("id")
    .reset_index(drop=True)
)

assert not set(shard_frame["id"]) & pilot_ids
assert not set(shard_frame["id"]) & holdout_ids
assert not set(shard_frame["template_key"]) & holdout_templates
assert shard_frame["id"].is_unique

shard_frame.to_csv(SHARD_INPUT_PATH, index=False, encoding="utf-8")
distribution = remaining["shard_index"].value_counts().sort_index().to_dict()
print("[DATA] clean:", len(clean))
print("[DATA] holdout IDs/templates:", len(holdout_ids), len(holdout_templates))
print("[DATA] training pool:", len(training_pool))
print("[DATA] reused pilot IDs:", len(pilot_ids))
print("[DATA] remaining:", len(remaining))
print("[DATA] shard distribution:", distribution)
print(f"[DATA] this shard {SHARD_INDEX}:", len(shard_frame), "->", SHARD_INPUT_PATH)

In [ ]:
# Cell 5 — Load frozen raw Qwen with vLLM continuous batching on A100.
# vLLM uses most VRAM as KV cache and dynamically keeps many decoding sequences active.
import ctypes
import glob
import site
import subprocess
import sys

# Colab's driver can support CUDA 13 while its system image omits libcudart.so.13.
# Preload the PyPI redistributable globally before importing vLLM's compiled extension.
runtime_candidates = []
nvrtc_candidates = []
for site_dir in site.getsitepackages():
    runtime_candidates.extend(glob.glob(
        str(Path(site_dir) / "nvidia" / "cu13" / "lib" / "libcudart.so.13*")
    ))
    runtime_candidates.extend(glob.glob(
        str(Path(site_dir) / "nvidia" / "cuda_runtime" / "lib" / "libcudart.so.13*")
    ))
    nvrtc_candidates.extend(glob.glob(
        str(Path(site_dir) / "nvidia" / "cu13" / "lib" / "libnvrtc.so.13*")
    ))
    nvrtc_candidates.extend(glob.glob(
        str(Path(site_dir) / "nvidia" / "cuda_nvrtc" / "lib" / "libnvrtc.so.13*")
    ))
runtime_candidates = sorted(set(runtime_candidates))
nvrtc_candidates = sorted(set(nvrtc_candidates))
assert runtime_candidates, (
    "libcudart.so.13 not found after Cell 1. Restart runtime once and rerun Cell 2+."
)
ctypes.CDLL(runtime_candidates[0], mode=ctypes.RTLD_GLOBAL)
print("[CUDA] preloaded:", runtime_candidates[0])
assert nvrtc_candidates, (
    "libnvrtc.so.13 not found after Cell 1. Install nvidia-cuda-nvrtc==13.0.88, "
    "restart once, then rerun Cell 2+."
)
ctypes.CDLL(nvrtc_candidates[0], mode=ctypes.RTLD_GLOBAL)
print("[CUDA] preloaded:", nvrtc_candidates[0])

# vLLM inspects model classes in a fresh Python subprocess. ctypes preload applies
# only to this process, so also export library directories for every child process.
cuda_library_dirs = sorted({
    str(Path(runtime_candidates[0]).parent),
    str(Path(nvrtc_candidates[0]).parent),
})
previous_ld_path = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(
    cuda_library_dirs + ([previous_ld_path] if previous_ld_path else [])
)
os.environ["LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]
print("[CUDA] child LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])

# Fail early with the complete child stderr before the expensive engine starts.
architecture_test = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "from vllm.model_executor.models.qwen2 "
            "import Qwen2ForCausalLM; "
            "print('Qwen2ForCausalLM inspection import OK')"
        ),
    ],
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)
if architecture_test.stdout.strip():
    print("[VLLM CHILD]", architecture_test.stdout.strip())
if architecture_test.returncode != 0:
    raise RuntimeError(
        "vLLM Qwen2 architecture subprocess failed:\n"
        + architecture_test.stderr
    )
print("[VLLM CHILD] architecture inspection passed")

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

torch.backends.cuda.matmul.allow_tf32 = True

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=MODEL_REVISION, use_fast=True, token=False
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

print("[VLLM] loading:", BASE_MODEL, flush=True)
llm = LLM(
    model=BASE_MODEL,
    revision=MODEL_REVISION,
    dtype="bfloat16",
    tensor_parallel_size=1,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=VLLM_MAX_MODEL_LEN,
    max_num_seqs=MAX_NUM_SEQS,
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
    enable_chunked_prefill=True,
    enable_prefix_caching=True,
    performance_mode="throughput",
    trust_remote_code=False,
)

# A strict terminal box, with commas permitted in the integer itself.
ANY_BOXED_RE = re.compile(r"\\boxed\s*\{\s*(-?\d(?:[\d,]*\d)?)\s*\}")

def strict_terminal_boxed(text):
    text = str(text or "")
    matches = list(ANY_BOXED_RE.finditer(text))
    if not matches:
        return None
    match = matches[-1]
    tail = text[match.end():].strip()
    # Permit normal TeX closers and terminal punctuation only.
    while True:
        before = tail
        tail = re.sub(r"^(?:\\\)|\\\]|\$\$|\$)", "", tail).strip()
        if tail == before:
            break
    tail = re.sub(r"^[.!]+$", "", tail).strip()
    if tail:
        return None
    return str(int(match.group(1).replace(",", "")))

def build_prompt(question):
    user_content = f"{str(question).strip()}\n\n{USER_SUFFIX}"
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

def load_valid_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows, malformed = [], 0
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                malformed += 1
    if malformed:
        print(f"[WARN] skipped malformed lines: {malformed}")
    return rows

def done_ids(path):
    return {str(row["id"]) for row in load_valid_jsonl(path) if "id" in row}

def append_jsonl(path, rows):
    with Path(path).open("a", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def generate_records(batch, batch_ordinal):
    prompts = [build_prompt(question) for question in batch["question"].tolist()]
    prompt_token_counts = [
        len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
        for prompt in prompts
    ]
    longest_prompt = max(prompt_token_counts, default=0)
    assert longest_prompt + EXTENDED_MAX_NEW_TOKENS <= VLLM_MAX_MODEL_LEN, (
        f"Prompt too long for untruncated generation: {longest_prompt} + "
        f"{EXTENDED_MAX_NEW_TOKENS} > {VLLM_MAX_MODEL_LEN}"
    )

    initial_seed = SEED + SHARD_INDEX * 1_000_000 + int(batch_ordinal)
    initial_params = SamplingParams(
        n=N_SAMPLES,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=INITIAL_MAX_NEW_TOKENS,
        seed=initial_seed,
    )
    outputs = llm.generate(prompts, initial_params, use_tqdm=False)

    records = []
    capped_positions = []
    capped_prompts = []
    for row_index, ((_, row), request_output) in enumerate(zip(batch.iterrows(), outputs)):
        candidates = []
        assert len(request_output.outputs) == N_SAMPLES
        for sample_index, completion in enumerate(request_output.outputs):
            raw = completion.text
            token_count = len(completion.token_ids)
            finish_reason = str(completion.finish_reason or "")
            capped = finish_reason == "length" or token_count >= INITIAL_MAX_NEW_TOKENS - 1
            candidates.append({
                "sample_index": sample_index,
                "raw_output": raw,
                "terminal_boxed_answer": strict_terminal_boxed(raw),
                "generated_tokens": int(token_count),
                "initial_finish_reason": finish_reason,
                "hit_initial_2048_cap": bool(capped),
                "redecoded_at_4096": False,
            })
            if capped:
                capped_positions.append((row_index, sample_index))
                capped_prompts.append(prompts[row_index])
        records.append({
            "id": str(row["id"]),
            "official_answer": str(row["answer"]),
            "shard_index": SHARD_INDEX,
            "num_shards": NUM_SHARDS,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "prompt_version": PROMPT_VERSION,
            "candidates": candidates,
        })

    # Only capped candidates are regenerated from the original prompt with a 4096-token cap.
    # This is a fresh replacement sample, explicitly recorded, never a hidden continuation.
    if capped_prompts:
        extended_params = SamplingParams(
            n=1,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=EXTENDED_MAX_NEW_TOKENS,
            seed=initial_seed + 500_000_000,
        )
        extended_outputs = llm.generate(
            capped_prompts, extended_params, use_tqdm=False
        )
        for (row_index, sample_index), request_output in zip(
            capped_positions, extended_outputs
        ):
            completion = request_output.outputs[0]
            raw = completion.text
            token_count = len(completion.token_ids)
            records[row_index]["candidates"][sample_index] = {
                "sample_index": sample_index,
                "raw_output": raw,
                "terminal_boxed_answer": strict_terminal_boxed(raw),
                "generated_tokens": int(token_count),
                "initial_finish_reason": "length",
                "extended_finish_reason": str(completion.finish_reason or ""),
                "hit_initial_2048_cap": True,
                "redecoded_at_4096": True,
                "hit_extended_4096_cap": bool(
                    str(completion.finish_reason or "") == "length"
                    or token_count >= EXTENDED_MAX_NEW_TOKENS - 1
                ),
            }
    return records

print("[VLLM] loaded")
print("[VLLM] gpu_memory_utilization:", GPU_MEMORY_UTILIZATION)
print("[VLLM] max_num_seqs / batched_tokens:", MAX_NUM_SEQS, MAX_NUM_BATCHED_TOKENS)

In [ ]:
# Cell 6 — Generate this shard. Resume-safe; rerun with the same SHARD_INDEX after interruption.
completed_ids = done_ids(RAW_ROLLOUT_PATH)
pending = shard_frame.loc[~shard_frame["id"].isin(completed_ids)].copy()
print(f"[ROLLOUT {SHARD_TAG}] done={len(completed_ids)} pending={len(pending)}", flush=True)

started = time.time()
initial_done = len(completed_ids)
total_batches = math.ceil(len(pending) / CHUNK_PROMPTS) if len(pending) else 0

for local_start in range(0, len(pending), CHUNK_PROMPTS):
    batch = pending.iloc[local_start:local_start + CHUNK_PROMPTS]
    # Derive the ordinal from the first row's fixed position in shard_frame, not pending order.
    first_id = str(batch.iloc[0]["id"])
    fixed_position = int(shard_frame.index[shard_frame["id"] == first_id][0])
    batch_ordinal = fixed_position // CHUNK_PROMPTS
    records = generate_records(batch, batch_ordinal)
    append_jsonl(RAW_ROLLOUT_PATH, records)

    new_done = min(local_start + len(batch), len(pending))
    completed = initial_done + new_done
    elapsed = time.time() - started
    rate = new_done / elapsed if elapsed else 0.0
    remaining_rows = len(shard_frame) - completed
    eta_minutes = remaining_rows / rate / 60 if rate else 0.0
    free_bytes, _ = torch.cuda.mem_get_info()
    print(
        f"[ROLLOUT {SHARD_TAG}] batch={local_start // CHUNK_PROMPTS + 1}/{total_batches} "
        f"rows={completed}/{len(shard_frame)} rate={rate:.3f}q/s "
        f"eta={eta_minutes:.1f}m gpu_free={free_bytes/1024**3:.1f}GiB",
        flush=True,
    )

raw_records = load_valid_jsonl(RAW_ROLLOUT_PATH)
raw_by_id = {str(row["id"]): row for row in raw_records}
assert len(raw_by_id) == len(shard_frame), (
    f"Incomplete/corrupt shard: {len(raw_by_id)}/{len(shard_frame)}"
)
assert set(raw_by_id) == set(shard_frame["id"])
print("[ROLLOUT] complete:", RAW_ROLLOUT_PATH)
print("[ROLLOUT] sha256:", sha256_file(RAW_ROLLOUT_PATH))

In [ ]:
# Cell 7 — Strict filter and select at most two correct, distinct solutions per problem.
MIN_VISIBLE_WORDS = 20
MAX_VISIBLE_WORDS = 350
MAX_TOTAL_TOKENS = 2048
MAX_PATHS_PER_QUESTION = 2

REJECT_PATTERNS = {
    "self_contradiction": re.compile(
        r"\b(?:wait|re[- ]?examining|i made (?:an|a) (?:error|mistake)|"
        r"this (?:is|was) (?:wrong|incorrect)|contradiction)\b", re.I
    ),
    "uncertainty": re.compile(
        r"\b(?:cannot determine|not enough information|insufficient information|"
        r"unable to solve|no unique answer|ambiguous)\b", re.I
    ),
    "unsupported_tool": re.compile(
        r"\b(?:subprocess|requests\.|urllib|read_csv|http://|https://)\b", re.I
    ),
    "teacher_reference_meta": re.compile(
        r"\b(?:provided solution|reference solution|private reference|teacher hint)\b", re.I
    ),
}

def path_signature(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    boxed = list(ANY_BOXED_RE.finditer(text))
    if boxed:
        text = text[:boxed[-1].start()]
    text = re.sub(r"\d+(?:\.\d+)?", "#", text)
    text = re.sub(r"[^a-z가-힣#+*/=<>\\]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

question_map = shard_frame.set_index("id")["question"].to_dict()
audit_rows, selected_rows = [], []
success_histogram = Counter()

for qid in shard_frame["id"]:
    record = raw_by_id[qid]
    official = str(record["official_answer"])
    accepted = []
    seen_signatures = set()

    strict_success_count = sum(
        strict_terminal_boxed(candidate["raw_output"]) == official
        for candidate in record["candidates"]
    )
    success_histogram[int(strict_success_count)] += 1

    for candidate in record["candidates"]:
        raw = str(candidate["raw_output"])
        terminal = strict_terminal_boxed(raw)
        word_count = len(raw.split())
        total_tokens = len(tokenizer(
            build_prompt(question_map[qid]) + raw,
            add_special_tokens=False,
        )["input_ids"])
        signature = path_signature(raw)
        reasons = []

        if terminal != official:
            reasons.append("terminal_boxed_answer_mismatch")
        if word_count < MIN_VISIBLE_WORDS:
            reasons.append("solution_too_short")
        if word_count > MAX_VISIBLE_WORDS:
            reasons.append("solution_too_long")
        if total_tokens > MAX_TOTAL_TOKENS:
            reasons.append("total_tokens_gt_2048")
        if candidate.get("hit_extended_4096_cap", False):
            reasons.append("generation_hit_4096_cap")
        for reason, pattern in REJECT_PATTERNS.items():
            if pattern.search(raw):
                reasons.append(reason)
        if not signature:
            reasons.append("empty_path_signature")
        elif signature in seen_signatures:
            reasons.append("duplicate_reasoning_path")

        decision = "accept" if not reasons else "reject"
        audit_rows.append({
            "id": qid,
            "sample_index": candidate["sample_index"],
            "official_answer": official,
            "terminal_boxed_answer": terminal or "",
            "word_count": word_count,
            "total_tokens": total_tokens,
            "generated_tokens": candidate["generated_tokens"],
            "hit_initial_2048_cap": candidate.get("hit_initial_2048_cap", False),
            "redecoded_at_4096": candidate.get("redecoded_at_4096", False),
            "hit_extended_4096_cap": candidate.get("hit_extended_4096_cap", False),
            "strict_success_count_of_4": strict_success_count,
            "decision": decision,
            "reasons": " | ".join(reasons),
        })

        if decision == "accept":
            seen_signatures.add(signature)
            accepted.append({
                "id": qid,
                "question": question_map[qid],
                "answer": official,
                "solution": raw,
                "word_count": word_count,
                "total_tokens": total_tokens,
                "sample_index": candidate["sample_index"],
                "success_count_of_4": strict_success_count,
                "temperature": TEMPERATURE,
                "source": "raw_qwen_self_rollout_r1",
                "prompt_version": PROMPT_VERSION,
                "shard_index": SHARD_INDEX,
            })

    # Short, complete traces first; unique signatures were enforced above.
    accepted.sort(key=lambda item: (item["total_tokens"], item["sample_index"]))
    selected_rows.extend(accepted[:MAX_PATHS_PER_QUESTION])

audit_df = pd.DataFrame(audit_rows)
selected_df = pd.DataFrame(selected_rows)
audit_df.to_csv(SHARD_AUDIT_PATH, index=False, encoding="utf-8")
selected_df.to_csv(SHARD_VERIFIED_PATH, index=False, encoding="utf-8")

accepted_ids = set(selected_df["id"]) if len(selected_df) else set()
report = {
    "run_id": RUN_ID,
    "shard_index": SHARD_INDEX,
    "num_shards": NUM_SHARDS,
    "input_rows": len(shard_frame),
    "raw_candidates": len(shard_frame) * N_SAMPLES,
    "strict_success_count_histogram": dict(sorted(success_histogram.items())),
    "questions_with_selected_trace": len(accepted_ids),
    "question_coverage": len(accepted_ids) / len(shard_frame) if len(shard_frame) else 0.0,
    "selected_traces": len(selected_df),
    "reject_reason_counts": dict(Counter(
        reason
        for value in audit_df.loc[audit_df["reasons"].ne(""), "reasons"]
        for reason in value.split(" | ")
    )),
    "generation": {
        "model": BASE_MODEL,
        "revision": MODEL_REVISION,
        "n": N_SAMPLES,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "initial_max_new_tokens": INITIAL_MAX_NEW_TOKENS,
        "extended_max_new_tokens": EXTENDED_MAX_NEW_TOKENS,
        "max_model_len": VLLM_MAX_MODEL_LEN,
        "engine": "vllm_continuous_batching",
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
        "max_num_seqs": MAX_NUM_SEQS,
        "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
        "prompt_version": PROMPT_VERSION,
    },
    "artifacts": {
        "input": str(SHARD_INPUT_PATH),
        "input_sha256": sha256_file(SHARD_INPUT_PATH),
        "raw": str(RAW_ROLLOUT_PATH),
        "raw_sha256": sha256_file(RAW_ROLLOUT_PATH),
        "audit": str(SHARD_AUDIT_PATH),
        "audit_sha256": sha256_file(SHARD_AUDIT_PATH),
        "verified": str(SHARD_VERIFIED_PATH),
        "verified_sha256": sha256_file(SHARD_VERIFIED_PATH),
    },
    "official_evaluation_files_read": [],
}
SHARD_REPORT_PATH.write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(report, ensure_ascii=False, indent=2))
print("[SHARD DONE]", SHARD_VERIFIED_PATH)

In [ ]:
# Cell 8 — Release GPU after this shard is safely stored.
import gc
del llm
gc.collect()
torch.cuda.empty_cache()
print("[GPU] model released; shard artifacts are persisted on Drive.")

## Final merge — run once, only after all three sessions finish

아래 셀은 세 shard의 raw/verified/report가 모두 완성된 후 한 세션에서만 실행합니다.
`RUN_FINAL_MERGE=True`로 바꾸기 전에는 아무 것도 병합하지 않습니다.

In [ ]:
# Cell 9 — Merge the completed 3 shards with the immutable 2K pilot.
RUN_FINAL_MERGE = False

if not RUN_FINAL_MERGE:
    print("[MERGE] disabled. Set RUN_FINAL_MERGE=True after all three shards finish.")
else:
    shard_verified_paths = [
        DATA_DIR / f"r1_native_verified_shard_{idx:02d}_of_{NUM_SHARDS:02d}.csv"
        for idx in range(NUM_SHARDS)
    ]
    shard_raw_paths = [
        CAND_DIR / f"r1_native_rollouts_shard_{idx:02d}_of_{NUM_SHARDS:02d}.jsonl"
        for idx in range(NUM_SHARDS)
    ]
    shard_report_paths = [
        REPORT_DIR / f"rollout_report_shard_{idx:02d}_of_{NUM_SHARDS:02d}.json"
        for idx in range(NUM_SHARDS)
    ]
    missing = [
        str(path) for path in shard_verified_paths + shard_raw_paths + shard_report_paths
        if not path.exists()
    ]
    assert not missing, "Missing shard artifacts:\n" + "\n".join(missing)

    pilot_verified = read_csv_strings(PILOT_VERIFIED_PATH)
    assert {"id", "question", "answer", "solution"}.issubset(pilot_verified.columns)
    pilot_verified = pilot_verified[["id", "question", "answer", "solution"]].copy()
    pilot_verified["source"] = "raw_qwen_self_rollout_r1_pilot_k8"
    pilot_verified["origin_shard"] = "pilot_2000"

    shard_frames = []
    for idx, path in enumerate(shard_verified_paths):
        frame = read_csv_strings(path)
        assert {"id", "question", "answer", "solution"}.issubset(frame.columns), path
        frame = frame[["id", "question", "answer", "solution", "source"]].copy()
        frame["origin_shard"] = f"shard_{idx:02d}"
        shard_frames.append(frame)

    merged = pd.concat([pilot_verified] + shard_frames, ignore_index=True)
    merged["answer"] = merged["answer"].map(normalize_integer)
    assert merged["answer"].notna().all()
    merged["template_key"] = merged["question"].map(normalized_template)

    # Exact solution dedup and final per-question cap. No question should exceed two rows.
    merged["solution_key"] = merged["solution"].map(
        lambda text: re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(text))).strip()
    )
    merged = merged.drop_duplicates(["id", "solution_key"], keep="first")
    merged["solution_words"] = merged["solution"].map(lambda text: len(str(text).split()))
    merged = (
        merged.sort_values(["id", "solution_words", "origin_shard"])
        .groupby("id", group_keys=False)
        .head(2)
        .reset_index(drop=True)
    )

    expected_training_ids = set(training_pool["id"])
    merged_ids = set(merged["id"])
    assert merged_ids.issubset(expected_training_ids)
    assert not merged_ids & holdout_ids
    assert not set(merged["template_key"]) & holdout_templates
    assert merged.groupby("id").size().max() <= 2
    assert merged["answer"].str.fullmatch(r"-?\d+").all()

    FINAL_R1_PATH = DATA_DIR / "r1_native_verified_full.csv"
    FINAL_AUDIT_PATH = DATA_DIR / "r1_native_coverage_all_training_ids.csv"
    FINAL_MANIFEST_PATH = REPORT_DIR / "r1_native_manifest.json"

    output_columns = ["id", "question", "answer", "solution", "source", "origin_shard"]
    merged[output_columns].to_csv(FINAL_R1_PATH, index=False, encoding="utf-8")

    coverage = training_pool[["id", "question", "answer", "template_key"]].copy()
    counts = merged.groupby("id").size().rename("selected_trace_count")
    coverage = coverage.merge(counts, left_on="id", right_index=True, how="left")
    coverage["selected_trace_count"] = coverage["selected_trace_count"].fillna(0).astype(int)
    coverage["has_selected_trace"] = coverage["selected_trace_count"].gt(0)
    coverage.to_csv(FINAL_AUDIT_PATH, index=False, encoding="utf-8")

    manifest = {
        "run_id": RUN_ID,
        "objective": "Champion-style R1 native self-RFT: raw Qwen k4, strict correct, max2.",
        "base_model": BASE_MODEL,
        "prompt_version": PROMPT_VERSION,
        "source_clean_train": str(CLEAN_TRAIN_PATH),
        "source_clean_sha256": sha256_file(CLEAN_TRAIN_PATH),
        "source_split_manifest": str(SOURCE_SPLIT_DIR / "split_manifest.json"),
        "source_split_manifest_sha256": sha256_file(SOURCE_SPLIT_DIR / "split_manifest.json"),
        "pilot_run": str(PILOT_RUN_DIR),
        "pilot_verified_sha256": sha256_file(PILOT_VERIFIED_PATH),
        "training_pool_rows": len(training_pool),
        "pilot_question_rows": len(pilot_index),
        "remaining_question_rows": len(remaining),
        "selected_question_rows": int(coverage["has_selected_trace"].sum()),
        "question_coverage": float(coverage["has_selected_trace"].mean()),
        "selected_trace_rows": len(merged),
        "trace_count_histogram": {
            str(key): int(value)
            for key, value in coverage["selected_trace_count"].value_counts().sort_index().items()
        },
        "generation": {
            "remaining_n": N_SAMPLES,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "initial_max_new_tokens": INITIAL_MAX_NEW_TOKENS,
            "extended_max_new_tokens": EXTENDED_MAX_NEW_TOKENS,
            "max_model_len": VLLM_MAX_MODEL_LEN,
            "engine": "vllm_continuous_batching",
            "pilot_reused": True,
        },
        "shards": [json.loads(path.read_text(encoding="utf-8")) for path in shard_report_paths],
        "artifacts": {
            "r1_training_csv": str(FINAL_R1_PATH),
            "r1_training_sha256": sha256_file(FINAL_R1_PATH),
            "coverage_csv": str(FINAL_AUDIT_PATH),
            "coverage_sha256": sha256_file(FINAL_AUDIT_PATH),
        },
        "official_evaluation_files_read": [],
    }
    FINAL_MANIFEST_PATH.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps({
        "training_pool_rows": manifest["training_pool_rows"],
        "selected_question_rows": manifest["selected_question_rows"],
        "question_coverage": manifest["question_coverage"],
        "selected_trace_rows": manifest["selected_trace_rows"],
        "trace_count_histogram": manifest["trace_count_histogram"],
        "r1_training_csv": str(FINAL_R1_PATH),
        "manifest": str(FINAL_MANIFEST_PATH),
    }, ensure_ascii=False, indent=2))

In [ ]:
# Cell 10 — Optional integrity summary. Run after Cell 9 merge.
if RUN_FINAL_MERGE:
    verified = read_csv_strings(FINAL_R1_PATH)
    assert list(verified.columns) == [
        "id", "question", "answer", "solution", "source", "origin_shard"
    ]
    assert verified["id"].isin(set(training_pool["id"])).all()
    assert verified["answer"].str.fullmatch(r"-?\d+").all()
    assert verified.groupby("id").size().max() <= 2
    print("[FINAL READY] rows:", len(verified))
    print("[FINAL READY] questions:", verified["id"].nunique())
    print("[FINAL READY] sha256:", sha256_file(FINAL_R1_PATH))
    print("[NEXT] Use this CSV for the R1 LoRA training notebook.")